# Section 7-11: Differential Privacy Track

## Overview

This notebook covers the Differential Privacy (DP) track of the AI Privacy module. We'll explore:

- **Section 7:** Differential Privacy Fundamentals
- **Section 8:** Opacus Library Setup
- **Section 9:** DP-SGD Setup Overview
- **Section 10:** Training Models and Measuring Privacy
- **Section 11:** Analyzing the Privacy-Utility Tradeoff
- **Section 12:** DP-SGD Challenge (separate)

### Key Concepts

1. **Differential Privacy (DP):** A formal framework for quantifying privacy loss
2. **DP-SGD:** Modified training procedure with gradient clipping and noise addition
3. **Privacy Budget (ε, δ):** Parameters controlling privacy strength
4. **Privacy-Utility Tradeoff:** Balancing privacy protection with model accuracy

### Dataset

We use CIFAR-10:
- 50,000 training images (10 classes)
- 10,000 test images
- Image size: 32×32 pixels
- Classes: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck

## Imports and Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

## CIFAR-10 Dataset Preparation

In [ ]:
# Define transforms
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

# Load CIFAR-10 dataset
print("Loading CIFAR-10 dataset...")
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

# Create data loaders
batch_size = 256
train_loader = DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=0)

print(f"Training set size: {len(trainset)}")
print(f"Test set size: {len(testset)}")
print(f"Number of classes: {len(trainset.classes)}")
print(f"Classes: {trainset.classes}")

## CNN Architecture for CIFAR-10

In [ ]:
class CIFAR10_CNN(nn.Module):
    """Simple CNN for CIFAR-10 classification"""
    
    def __init__(self, num_classes=10):
        super(CIFAR10_CNN, self).__init__()
        
        # Convolutional layers
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        
        # Pooling and batch normalization
        self.pool = nn.MaxPool2d(2, 2)
        self.bn1 = nn.BatchNorm2d(32)
        self.bn2 = nn.BatchNorm2d(64)
        self.bn3 = nn.BatchNorm2d(128)
        
        # Fully connected layers
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, num_classes)
        
        self.dropout = nn.Dropout(0.5)
    
    def forward(self, x):
        # Block 1: Conv -> BN -> ReLU -> Pool
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.pool(x)  # 32 -> 16
        
        # Block 2: Conv -> BN -> ReLU -> Pool
        x = self.conv2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.pool(x)  # 16 -> 8
        
        # Block 3: Conv -> BN -> ReLU -> Pool
        x = self.conv3(x)
        x = self.bn3(x)
        x = F.relu(x)
        x = self.pool(x)  # 8 -> 4
        
        # Flatten and fully connected layers
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout(x)
        
        x = self.fc2(x)
        x = F.relu(x)
        x = self.dropout(x)
        
        x = self.fc3(x)
        return x

# Create model
model = CIFAR10_CNN(num_classes=10).to(device)
print(f"Model created and moved to {device}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

## Training Function

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, labels in tqdm(train_loader, desc="Training", leave=False):
        inputs, labels = inputs.to(device), labels.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Statistics
        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    avg_loss = total_loss / len(train_loader)
    accuracy = 100 * correct / total
    
    return avg_loss, accuracy

def test(model, test_loader, criterion, device):
    """Test the model"""
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc="Testing", leave=False):
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    avg_loss = total_loss / len(test_loader)
    accuracy = 100 * correct / total
    
    return avg_loss, accuracy

print("Training and test functions defined")

## Differential Privacy Concepts Visualization

In [ ]:
# Visualize the differential privacy concepts
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Differential Privacy Fundamentals', fontsize=16, fontweight='bold')

# 1. Privacy Budget (epsilon) impact
epsilon_values = np.array([0.1, 0.5, 1, 3, 10, 100])
exp_epsilon = np.exp(epsilon_values)

axes[0, 0].bar(range(len(epsilon_values)), exp_epsilon, color='steelblue', alpha=0.7, edgecolor='black')
axes[0, 0].set_xticks(range(len(epsilon_values)))
axes[0, 0].set_xticklabels([f'ε={e}' for e in epsilon_values])
axes[0, 0].set_ylabel('e^ε (Multiplicative Bound)', fontsize=11)
axes[0, 0].set_title('Privacy Budget vs Output Likelihood Bound\n(Lower ε = Stronger Privacy)', fontsize=11)
axes[0, 0].set_yscale('log')
axes[0, 0].grid(axis='y', alpha=0.3)

# 2. Gradient Clipping Visualization
grad_norms = np.array([0.3, 0.7, 0.95, 1.2, 1.5, 2.0, 3.5, 5.2])
max_grad_norm = 1.0
clipped_norms = np.minimum(grad_norms, max_grad_norm)

x_pos = np.arange(len(grad_norms))
axes[0, 1].bar(x_pos, grad_norms, label='Original Gradient Norm', alpha=0.6, color='coral', edgecolor='black')
axes[0, 1].bar(x_pos, clipped_norms, label='Clipped Gradient Norm', alpha=0.8, color='darkred', edgecolor='black')
axes[0, 1].axhline(y=max_grad_norm, color='red', linestyle='--', linewidth=2, label=f'Clipping Threshold={max_grad_norm}')
axes[0, 1].set_ylabel('Norm Value', fontsize=11)
axes[0, 1].set_xlabel('Sample Index', fontsize=11)
axes[0, 1].set_title('Gradient Clipping: Bounding Sensitivity', fontsize=11)
axes[0, 1].legend(loc='upper left', fontsize=9)
axes[0, 1].grid(axis='y', alpha=0.3)

# 3. Noise Addition for Different Privacy Levels
epsilon_configs = [10, 3, 1]
noise_multipliers = [1.2, 3.8, 10.0]  # Approximate values for CIFAR-10
max_grad_norm_val = 1.0
noise_stds = [max_grad_norm_val * nm for nm in noise_multipliers]

colors = ['green', 'orange', 'red']
x_noise = np.linspace(-5, 5, 1000)

for i, (eps, noise_std, color) in enumerate(zip(epsilon_configs, noise_stds, colors)):
    noise_dist = (1 / (noise_std * np.sqrt(2 * np.pi))) * np.exp(-(x_noise ** 2) / (2 * noise_std ** 2))
    axes[1, 0].plot(x_noise, noise_dist, label=f'ε={eps} (σ={noise_std:.1f})', linewidth=2.5, color=color)

axes[1, 0].set_xlabel('Noise Value', fontsize=11)
axes[1, 0].set_ylabel('Probability Density', fontsize=11)
axes[1, 0].set_title('Gaussian Noise Addition\n(Higher Privacy = More Noise)', fontsize=11)
axes[1, 0].legend(fontsize=10)
axes[1, 0].grid(alpha=0.3)

# 4. Privacy-Utility Tradeoff Illustration
epsilon_range = np.array([0.5, 1, 2, 3, 5, 8, 10, 15, 20])
# Simulated relationship: higher epsilon -> less privacy loss -> higher utility
utility_scores = 100 * (1 - np.exp(-0.3 * epsilon_range))
privacy_strength = 100 / (1 + 0.15 * epsilon_range)

axes[1, 1].plot(epsilon_range, utility_scores, 'o-', linewidth=2.5, markersize=8, label='Model Utility (Accuracy)', color='green')
axes[1, 1].plot(epsilon_range, privacy_strength, 's-', linewidth=2.5, markersize=8, label='Privacy Strength', color='red')
axes[1, 1].axvline(x=3, color='blue', linestyle='--', alpha=0.7, linewidth=2, label='ε=3 (Strong Privacy)')
axes[1, 1].axvline(x=10, color='orange', linestyle='--', alpha=0.7, linewidth=2, label='ε=10 (Modest Privacy)')
axes[1, 1].set_xlabel('Privacy Budget (ε)', fontsize=11)
axes[1, 1].set_ylabel('Score (%)', fontsize=11)
axes[1, 1].set_title('Privacy-Utility Tradeoff', fontsize=11)
axes[1, 1].legend(fontsize=10)
axes[1, 1].grid(alpha=0.3)
axes[1, 1].set_ylim([0, 105])

plt.tight_layout()
plt.savefig('./output/07_dp_fundamentals_concepts.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nDifferential Privacy Concepts Visualized:")
print("1. Privacy Budget (ε) Impact: Shows how privacy strength increases with smaller ε")
print("2. Gradient Clipping: Bounds individual sample influence on gradients")
print("3. Gaussian Noise Addition: Adds noise calibrated to privacy budget")
print("4. Privacy-Utility Tradeoff: Stronger privacy requires sacrificing some accuracy")

## Privacy Accounting: Budget Composition Across Training Steps

In [ ]:
# Privacy accounting visualization
def compute_privacy_budget(num_epochs, batch_size, dataset_size, target_epsilon, delta=1e-5):
    """
    Estimate noise multiplier needed to achieve target epsilon
    Using simplified RDP accounting
    """
    num_steps = (dataset_size / batch_size) * num_epochs
    return num_steps

# Parameters
dataset_size = 50000  # CIFAR-10 training set
batch_size = 256
num_epochs = 10
target_epsilons = [10, 3, 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Privacy Budget Composition Across Training', fontsize=14, fontweight='bold')

# Calculate total steps
total_steps = compute_privacy_budget(num_epochs, batch_size, dataset_size, 10)
epochs_per_step = np.arange(1, num_epochs + 1)
steps_per_epoch = dataset_size / batch_size

# 1. Cumulative privacy loss with different epsilon values
for target_eps in target_epsilons:
    # Simplified: each step uses epsilon_per_step
    epsilon_per_step = target_eps / num_epochs
    cumulative_eps = epochs_per_step * epsilon_per_step
    
    axes[0].plot(epochs_per_step, cumulative_eps, 'o-', linewidth=2.5, markersize=8, 
                 label=f'Target ε={target_eps} (ε/step={epsilon_per_step:.3f})')

axes[0].set_xlabel('Training Epoch', fontsize=11)
axes[0].set_ylabel('Cumulative Privacy Loss (ε)', fontsize=11)
axes[0].set_title('How Privacy Budget Accumulates Over Training Steps', fontsize=11)
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# 2. Required noise multiplier for different configurations
epsilon_values = np.array([1, 3, 5, 8, 10])
noise_multipliers_est = [10.0, 3.8, 2.5, 1.5, 1.2]  # Estimated for CIFAR-10, batch_size=256

axes[1].plot(epsilon_values, noise_multipliers_est, 'o-', linewidth=2.5, markersize=10, 
             color='darkblue', label='Estimated Noise Multiplier')
axes[1].fill_between(epsilon_values, 0, noise_multipliers_est, alpha=0.3, color='lightblue')

# Annotations
for eps, nm in zip(epsilon_values, noise_multipliers_est):
    axes[1].annotate(f'σ={nm:.1f}', xy=(eps, nm), xytext=(5, 5), 
                    textcoords='offset points', fontsize=9, fontweight='bold')

axes[1].set_xlabel('Privacy Budget (ε)', fontsize=11)
axes[1].set_ylabel('Noise Multiplier (σ)', fontsize=11)
axes[1].set_title('Privacy vs Required Noise\n(Higher Privacy = Higher Noise = Harder Learning)', fontsize=11)
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('./output/07_privacy_accounting.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nPrivacy Accounting Summary (CIFAR-10):")
print(f"  Dataset size: {dataset_size:,}")
print(f"  Batch size: {batch_size}")
print(f"  Training epochs: {num_epochs}")
print(f"  Total training steps: {int(total_steps * num_epochs):,}")
print(f"  Steps per epoch: {int(steps_per_epoch):.0f}")
print(f"\nNoise Multiplier Requirements:")
for eps, nm in zip(epsilon_values, noise_multipliers_est):
    print(f"  ε = {eps:2d}: noise_multiplier ≈ {nm:4.1f}")

## Section 7 Summary

### Key Takeaways

1. **Differential Privacy Definition:** (ε, δ)-DP provides formal privacy guarantees by bounding how much any single record can affect the algorithm's output distribution.

2. **Privacy Parameters:**
   - **ε (epsilon):** Privacy budget - lower values = stronger privacy
     - ε = 1: Very strong privacy
     - ε = 3: Strong privacy (academic standard)
     - ε = 10: Modest privacy (practical baseline)
   - **δ (delta):** Failure probability - typically set to 10^-5 for large datasets

3. **DP-SGD Mechanism:** Three components:
   - **Gradient Clipping:** Bounds L2 norm to `max_grad_norm` (typically 1.0)
   - **Noise Addition:** Gaussian noise with σ = max_grad_norm × noise_multiplier
   - **Privacy Composition:** Tracks cumulative privacy loss across training steps

4. **Privacy-Utility Tradeoff:**
   - Stronger privacy (lower ε) requires higher noise
   - Higher noise makes optimization harder
   - Results in lower model accuracy

5. **Privacy Accounting:**
   - Privacy loss accumulates with each training step
   - Advanced composition (RDP) provides tighter bounds than naive addition
   - Opacus automatically computes required noise multiplier

### Next Steps

In the following sections, we'll:
- Install and configure Opacus (Section 8)
- Set up DP-SGD training (Section 9)
- Train privacy-preserving models (Section 10)
- Analyze privacy-utility tradeoffs (Section 11)
- Complete DP-SGD challenge (Section 12)